### Seq2Seq模型

In [ ]:
import jieba
import torch
import torch.nn as nn
from collections import Counter

from jieba import tokenize

data = [
    ("你好，今天天气真好！", "Hello, the weather is nice today!"),
    ("你吃饭了吗？", "Have you eaten yet?"),
    ("深度学习很有趣。", "Deep learning is interesting."),
    ("我们一起学习吧。", "We are learning together."),
    ("这是一个测试案例。", "This is a test example.")
]


def tokenize_chinese(text):
    return list(jieba.cut(text))


def tokenize_english(text):
    return text.lower().split()


chinese_vocab = [tokenize_chinese(pair[0]) for pair in data]
english_vocab = [tokenize_english(pair[1]) for pair in data]

chinese_sentences = [tokenize_chinese(pair[0]) for pair in data]
english_sentences = [tokenize_english(pair[1]) for pair in data]




In [ ]:
special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']


def build_vocab(sentences):
    counter = Counter()

    for sentence in sentences:
        for word in sentence:
            counter[word] += 1

    vocab = special_tokens.copy()
    for word, count in counter.items():
        if word not in special_tokens:
            vocab.append(word)

    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    return word_to_idx, vocab


chinese_word_to_idx, chinese_vocab = build_vocab([sentence for sentence in chinese_sentences])
english_word_to_idx, english_vocab = build_vocab([sentence for sentence in english_sentences])

print(chinese_vocab)
print(english_vocab)
print(chinese_word_to_idx)
print(english_word_to_idx)


In [ ]:
ch_vocab_size = len(chinese_vocab)
en_vocab_size = len(english_vocab)
hidden_size = 256
batch_size = 2
Learning_rate = 0.001


def tokenize(words, word_to_idx):
    return [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]


processed_data_ch = []
processed_data_en = []

for ch, en in zip(chinese_sentences, english_sentences):
    ch_numerical = [chinese_word_to_idx['<BOS>']] + tokenize(ch, chinese_word_to_idx) + [chinese_word_to_idx['<EOS>']]
    en_numerical = [english_word_to_idx['<BOS>']] + tokenize(en, english_word_to_idx) + [english_word_to_idx['<EOS>']]
    processed_data_ch.append(torch.LongTensor(ch_numerical))
    processed_data_en.append(torch.LongTensor(en_numerical))

print(processed_data_ch)
print(processed_data_en)


In [ ]:
from torch.nn.utils import rnn

processed_data_ch_pad = rnn.pad_sequence(processed_data_ch, batch_first=True,
                                         padding_value=chinese_word_to_idx['<PAD>'])
processed_data_en_pad = rnn.pad_sequence(processed_data_en, batch_first=True,
                                         padding_value=english_word_to_idx['<PAD>'])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(processed_data_ch_pad, processed_data_en_pad)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for src, trg in dataloader:
    print(src)
    print(trg)
    break

In [ ]:
from torch import optim
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, trg, hidden):
        embedded = self.embedding(trg)
        output, hidden = self.gru(embedded, hidden)
        output = self.fc(output)
        return output, hidden


encoder = Encoder(ch_vocab_size, hidden_size)
decoder = Decoder(en_vocab_size, hidden_size)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=Learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=english_word_to_idx['<PAD>'])

epochs = 100

for step in range(epochs):
    for input, target in dataloader:
        _, hidden = encoder(input)
        decoder_input = target[:, :-1]
        decoder_target = target[:, 1:]

        decoder_output, _ = decoder(decoder_input, hidden)
        loss=criterion(decoder_output.view(-1,decoder_output.size(-1)),decoder_target.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'epoch:{step+1},loss:{loss.item()}')

In [ ]:
def translate(sentence, encoder, decoder):
    token = tokenize_chinese(sentence)
    numerical = [chinese_word_to_idx.get(word, chinese_word_to_idx['<UNK>']) for word in token]
    numerical = [chinese_word_to_idx['<BOS>']] + numerical + [chinese_word_to_idx['<EOS>']]
    src = torch.LongTensor(numerical).unsqueeze(0)
    _, hidden = encoder(src)
    trg_indexes = [english_word_to_idx['<BOS>']]

    for _ in range(10):

        trg_tensor = torch.LongTensor([trg_indexes[-1]]).unsqueeze(0)
        with torch.no_grad():
            output, hidden = decoder(trg_tensor, hidden)
        pred_token = output.argmax().item()
        trg_indexes.append(pred_token)
        if pred_token == english_word_to_idx['<EOS>']:
            break
    return ' '.join(english_vocab[idx] for idx in trg_indexes[1:-1])

test_sentence='你好，今天天气真好！'

print(translate(test_sentence,encoder,decoder))